In [1]:
# ============================================================
# EVAL 3: Full λ-Sweep Training + Evaluation
# Run in Google Colab (T4 GPU, ~2-3 hrs per λ value)
# Checkpoints saved to Google Drive — safe to resume after disconnect
# ============================================================

# ── CELL 1: Mount Drive + Install ───────────────────────────
# from google.colab import drive
# drive.mount('/content/drive')

SAVE_DIR = "NLP_λ_Sweep_Checkpoints"
import os; os.makedirs(SAVE_DIR, exist_ok=True)

In [2]:
!pip install transformers peft datasets huggingface_hub bert-score rouge-score accelerate -q

In [18]:
# ── CELL 2: Imports ─────────────────────────────────────────
import json, random, os
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from bert_score import score as bert_score_fn
from rouge_score import rouge_scorer as rouge_lib
from huggingface_hub import hf_hub_download
from peft import LoraConfig, TaskType, get_peft_model
from tqdm import tqdm
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          T5ForConditionalGeneration, T5Tokenizer, AutoModelForCausalLM)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
torch.manual_seed(42)
random.seed(42)

Device: cuda


In [4]:
# ── CELL 3: Load HC3 full dataset ───────────────────────────
filepath = hf_hub_download(
    repo_id="Hello-SimpleAI/HC3",
    filename="all.jsonl",
    repo_type="dataset"
)

ai_texts, human_texts = [], []
with open(filepath) as f:
    for line in f:
        item = json.loads(line)
        for ans in item.get("chatgpt_answers", []):
            if ans and len(ans.strip()) > 50:
                ai_texts.append(ans.strip())
        for ans in item.get("human_answers", []):
            if ans and len(ans.strip()) > 50:
                human_texts.append(ans.strip())

random.shuffle(ai_texts)
random.shuffle(human_texts)

# Fixed splits — same seed as baselines
test_ai,  rest_ai  = ai_texts[:1000],    ai_texts[1000:]
test_hu,  rest_hu  = human_texts[:1000], human_texts[1000:]
val_ai,   train_ai = rest_ai[:500],      rest_ai[500:]

print(f"Train AI : {len(train_ai)}")
print(f"Val AI   : {len(val_ai)}")
print(f"Test AI  : {len(test_ai)}  |  Test Human: {len(test_hu)}")

Train AI : 25339
Val AI   : 500
Test AI  : 1000  |  Test Human: 1000


In [5]:
# ── CELL 4: Load frozen surrogate detector ───────────────────
DET_NAME = "Hello-SimpleAI/chatgpt-detector-roberta"
det_tok  = AutoTokenizer.from_pretrained(DET_NAME)
detector = AutoModelForSequenceClassification.from_pretrained(DET_NAME).to(device)
detector.eval()
for p in detector.parameters():
    p.requires_grad_(False)
print("RoBERTa surrogate loaded and frozen.")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 21456.70it/s]
RobertaForSequenceClassification LOAD REPORT from: Hello-SimpleAI/chatgpt-detector-roberta
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RoBERTa surrogate loaded and frozen.


In [6]:
# ── CELL 5: Build LoRA adapter on T5-base ───────────────────
def build_model(base="t5-base"):
    tokenizer  = T5Tokenizer.from_pretrained(base)
    base_model = T5ForConditionalGeneration.from_pretrained(base)
    lora_cfg   = LoraConfig(
        task_type=TaskType.SEQ_2_SEQ_LM,
        r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules=["q", "v"],
    )
    model = get_peft_model(base_model, lora_cfg).to(device)
    model.print_trainable_parameters()
    return model, tokenizer

In [7]:
# ── CELL 6: Loss functions ───────────────────────────────────
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

from transformers import AutoModel

# Load MiniLM ONCE — 22MB, stays frozen in VRAM
print("Loading MiniLM for semantic loss...")
sem_tok   = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
sem_model = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2").to(device)
sem_model.eval()
for p in sem_model.parameters():
    p.requires_grad_(False)
print("MiniLM loaded (22MB).")

def mean_pool(out, mask):
    t = out.last_hidden_state
    m = mask.unsqueeze(-1).expand(t.size()).float()
    return (t * m).sum(1) / m.sum(1).clamp(min=1e-9)

def compute_l_grad(token_logits):
    det_embed = detector.roberta.embeddings.word_embeddings.weight
    probs     = F.softmax(token_logits, dim=-1)
    min_v     = min(probs.shape[-1], det_embed.shape[0])
    pseudo    = torch.matmul(probs[:, :, :min_v], det_embed[:min_v])
    logits    = detector(inputs_embeds=pseudo).logits
    targets   = torch.zeros(logits.size(0), dtype=torch.long, device=device)
    return F.cross_entropy(logits, targets)

def compute_l_rl(model, tokenizer, texts, G=2, max_new=48):
    """Faster, NaN-safe GRPO."""
    loss_accum = torch.tensor(0.0, device=device)
    for text in texts[:1]:   # only 1 text per call, not 2
        inp = tokenizer(
            "paraphrase: " + text[:300],   # truncate long texts
            return_tensors="pt", truncation=True, max_length=96
        ).to(device)
        lps, rewards = [], []
        for _ in range(G):   # G=2 instead of 4
            with torch.no_grad():
                out = model.generate(
                    input_ids=inp["input_ids"],
                    attention_mask=inp["attention_mask"],
                    max_new_tokens=max_new,
                    do_sample=True, temperature=0.9,
                    output_scores=True,
                    return_dict_in_generate=True,
                    decoder_start_token_id=0
                )
            ids  = out.sequences[0][inp.input_ids.shape[1]:]
            if len(ids) == 0:
                continue
            # NaN-safe log prob
            lp = 0.0
            for i, s in enumerate(out.scores):
                if i >= len(ids): break
                token_lp = F.log_softmax(s, dim=-1)[0, ids[i]].item()
                if not (torch.isnan(torch.tensor(token_lp)) or
                        torch.isinf(torch.tensor(token_lp))):
                    lp += token_lp
            cand = tokenizer.decode(ids, skip_special_tokens=True)
            if not cand.strip():
                continue
            enc = det_tok(cand, return_tensors="pt",
                          truncation=True, max_length=96).to(device)
            with torch.no_grad():
                ai_p = torch.softmax(detector(**enc).logits, dim=-1)[0, 1].item()
            lps.append(lp)
            rewards.append(1.0 - ai_p)

        if len(lps) < 2:
            continue
        r_t  = torch.tensor(rewards, dtype=torch.float32, device=device)
        lp_t = torch.tensor(lps,     dtype=torch.float32, device=device)
        # Clamp log probs to prevent NaN
        lp_t = torch.clamp(lp_t, min=-50.0, max=0.0)
        adv  = (r_t - r_t.mean()) / (r_t.std() + 1e-8)
        loss_accum = loss_accum - (adv * lp_t).mean()

    # NaN guard
    if torch.isnan(loss_accum) or torch.isinf(loss_accum):
        return torch.tensor(0.0, device=device)
    return loss_accum

def compute_l_sem(originals, paraphrases):
    # MiniLM cosine similarity — no large model, no reloading
    enc_o = sem_tok(originals,   padding=True, truncation=True,
                    max_length=128, return_tensors="pt").to(device)
    enc_p = sem_tok(paraphrases, padding=True, truncation=True,
                    max_length=128, return_tensors="pt").to(device)
    with torch.no_grad():
        emb_o = mean_pool(sem_model(**enc_o), enc_o["attention_mask"])
        emb_p = mean_pool(sem_model(**enc_p), enc_p["attention_mask"])
    return (1.0 - F.cosine_similarity(emb_o, emb_p).mean()).to(device)

print("Loss functions ready.")

Loading MiniLM for semantic loss...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2145.72it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


MiniLM loaded (22MB).
Loss functions ready.


In [8]:
# ── CELL 7: Evaluation helpers ───────────────────────────────
def generate_paraphrases(model, tokenizer, texts, max_new=128, batch_size=8):
    model.eval()
    results = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc   = tokenizer(
            ["paraphrase: " + t for t in batch],
            return_tensors="pt", padding=True,
            truncation=True, max_length=256
        ).to(device)
        with torch.no_grad():
            ids = model.generate(
                input_ids=enc["input_ids"],
                attention_mask=enc["attention_mask"],
                max_new_tokens=max_new,
                do_sample=False,
                decoder_start_token_id=0
            )
        results += tokenizer.batch_decode(ids, skip_special_tokens=True)
    return results

def eval_asr(texts, batch_size=32):
    preds = []
    for i in range(0, len(texts), batch_size):
        enc = det_tok(texts[i:i+batch_size], return_tensors="pt",
                      padding=True, truncation=True, max_length=512).to(device)
        with torch.no_grad():
            preds += torch.argmax(detector(**enc).logits, dim=-1).cpu().tolist()
    return sum(p == 0 for p in preds) / len(preds) * 100

def eval_bertscore(originals, paraphrases):
    try:
        from bert_score import score as bs_fn
        _, _, F = bs_fn(
            paraphrases, originals,
            model_type="distilbert-base-uncased",  # lighter, avoids RoBERTa conflict
            device=device, verbose=False
        )
        return F.mean().item()
    except Exception as e:
        print(f"  BERTScore skipped: {e}")
        return -1.0   # placeholder, compute separately after sweep

def eval_rouge(originals, paraphrases):
    sc = rouge_lib.RougeScorer(["rougeL"], use_stemmer=True)
    return np.mean([sc.score(o, p)["rougeL"].fmeasure
                    for o, p in zip(originals, paraphrases)])

print("Eval helpers ready.")

Eval helpers ready.


In [9]:
# ── CELL 8: Training function (one λ) ───────────────────────
def train_one_lambda(
    lam,
    alpha        = 0.3,
    epochs       = 3,
    batch_size   = 4,    # ← reduced from 8
    rl_per_batch = 1,    # ← reduced from 2
    max_train    = 5000,
):
    lam_tag   = str(lam).replace(".", "_")
    save_path = f"{SAVE_DIR}/lambda_{lam_tag}"
    os.makedirs(save_path, exist_ok=True)

    done_file = f"{save_path}/DONE.json"
    if os.path.exists(done_file):
        print(f"λ={lam} already done — skipping.")
        with open(done_file) as f:
            return json.load(f)

    print(f"\n{'='*55}")
    print(f"  λ = {lam}   (gradient: {lam}   RL: {1-lam})")
    print(f"{'='*55}")

    model, tokenizer = build_model()
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad], lr=2e-4
    )

    train_data = train_ai[:max_train]
    epoch_loss = 0.0

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        random.shuffle(train_data)

        for i in tqdm(range(0, len(train_data), batch_size),
                      desc=f"  Epoch {epoch+1}/{epochs}  λ={lam}"):
            batch = train_data[i:i+batch_size]
            enc   = tokenizer(
                ["paraphrase: " + t for t in batch],
                return_tensors="pt", padding=True,
                truncation=True, max_length=128   # ← reduced from 256
            ).to(device)

            out = model(
                input_ids=enc["input_ids"],
                attention_mask=enc["attention_mask"],
                decoder_input_ids=enc["input_ids"],
            )

            loss = torch.tensor(0.0, device=device)

            if lam > 0:
                loss = loss + lam * compute_l_grad(out.logits)

            if lam < 1.0:
                loss = loss + (1 - lam) * compute_l_rl(
                    model, tokenizer, batch[:rl_per_batch]
                )

            # Semantic constraint — MiniLM, no reloading
            with torch.no_grad():
                gen_ids = model.generate(
                    input_ids=enc["input_ids"],
                    attention_mask=enc["attention_mask"],
                    max_new_tokens=64,    # ← reduced from 128
                    do_sample=False,
                    decoder_start_token_id=0
                )
            paras = tokenizer.batch_decode(gen_ids, skip_special_tokens=True)
            loss  = loss + alpha * compute_l_sem(batch, paras)

            optimizer.zero_grad()
            torch.cuda.empty_cache()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            epoch_loss += loss.item()

        n_batches  = max(1, len(train_data) // batch_size)
        epoch_loss = epoch_loss / n_batches
        print(f"  Epoch {epoch+1} | Loss: {epoch_loss:.4f}")

        ckpt = f"{save_path}/epoch_{epoch+1}"
        model.save_pretrained(ckpt)
        tokenizer.save_pretrained(ckpt)
        print(f"  Saved → {ckpt}")

    print(f"\n  Generating paraphrases on 200 test samples...")
    paras = generate_paraphrases(model, tokenizer, test_ai[:200])
    asr   = eval_asr(paras)
    bs    = eval_bertscore(test_ai[:200], paras)  # BERTScore only here
    rl    = eval_rouge(test_ai[:200], paras)

    pd.DataFrame({
        "id":            [f"hc3_ai_{i:05d}_evaded" for i in range(len(paras))],
        "text":          paras,
        "source":        "ai",
        "attack_type":   "gradient",
        "attack_owner":  "udaiveer",
        "generator_model": "gpt3.5-turbo",
        "original_text": test_ai[:200],
    }).to_csv(f"{save_path}/evaded.csv", index=False)

    results = {
        "lambda": lam, "asr": round(asr, 2),
        "bertscore_f1": round(bs, 4), "rouge_l": round(rl, 4),
        "loss_final": round(epoch_loss, 4),
    }
    with open(done_file, "w") as f:
        json.dump(results, f, indent=2)

    print(f"  λ={lam} DONE → ASR:{asr:.1f}%  BS:{bs:.4f}  RL:{rl:.4f}")
    return results

In [11]:
# ── CELL 9 PATCH: Handle λ=0.0 separately ───────────────────
# For pure RL, loss must be computed differently to maintain grad_fn

def train_one_lambda_pure_rl(
    alpha=0.3, epochs=3, batch_size=4, max_train=5000
):
    """λ=0.0: pure RL. Uses REINFORCE-style update, no gradient loss."""
    lam_tag   = "0_0"
    save_path = f"{SAVE_DIR}/lambda_{lam_tag}"
    os.makedirs(save_path, exist_ok=True)

    done_file = f"{save_path}/DONE.json"
    if os.path.exists(done_file):
        print("λ=0.0 already done — skipping.")
        with open(done_file) as f:
            return json.load(f)

    print(f"\n{'='*55}")
    print(f"  λ = 0.0   (gradient: 0.0   RL: 1.0)")
    print(f"{'='*55}")

    model, tokenizer = build_model()
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad], lr=2e-4
    )

    train_data = train_ai[:max_train]
    epoch_loss = 0.0

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        random.shuffle(train_data)

        for i in tqdm(range(0, len(train_data), batch_size),
                      desc=f"  Epoch {epoch+1}/{epochs}  λ=0.0"):
            batch = train_data[i:i+batch_size]

            # Pure RL: need differentiable log probs from the model directly
            loss = torch.tensor(0.0, device=device, requires_grad=False)
            total_loss_val = 0.0

            for text in batch[:1]:   # 1 text per batch for RL
                inp = tokenizer(
                    "paraphrase: " + text,
                    return_tensors="pt", truncation=True, max_length=96
                ).to(device)

                G = 2
                rewards, log_prob_tensors = [], []

                for _ in range(G):
                    # Forward pass through model to get logits (keeps grad)
                    dec_ids = torch.zeros(1, 1, dtype=torch.long, device=device)
                    generated_ids, token_log_probs = [], []

                    model_input = inp["input_ids"]
                    enc_out = model.model.encoder(
                        input_ids=model_input,
                        attention_mask=inp["attention_mask"]
                    )

                    cur_ids = dec_ids
                    for step in range(48):
                        out = model.model.decoder(
                            input_ids=cur_ids,
                            encoder_hidden_states=enc_out.last_hidden_state,
                            encoder_attention_mask=inp["attention_mask"]
                        )
                        logits = model.lm_head(out.last_hidden_state[:, -1, :])
                        probs  = torch.softmax(logits / 0.9, dim=-1)
                        token  = torch.multinomial(probs, 1)
                        lp     = torch.log(probs[0, token.item()] + 1e-8)
                        token_log_probs.append(lp)
                        generated_ids.append(token.item())
                        cur_ids = token
                        if token.item() == tokenizer.eos_token_id:
                            break

                    cand = tokenizer.decode(generated_ids, skip_special_tokens=True)
                    if cand.strip():
                        enc = det_tok(cand, return_tensors="pt",
                                      truncation=True, max_length=96).to(device)
                        with torch.no_grad():
                            ai_p = torch.softmax(detector(**enc).logits, dim=-1)[0, 1].item()
                        rewards.append(1.0 - ai_p)
                    else:
                        rewards.append(0.0)

                    if token_log_probs:
                        log_prob_tensors.append(torch.stack(token_log_probs).mean())

                if len(rewards) >= 2 and log_prob_tensors:
                    r_t  = torch.tensor(rewards, device=device)
                    adv  = (r_t - r_t.mean()) / (r_t.std() + 1e-8)
                    rl_loss = torch.tensor(0.0, device=device)
                    for a, lp in zip(adv, log_prob_tensors):
                        rl_loss = rl_loss - a * lp
                    loss = rl_loss / len(rewards)

            # Semantic constraint
            enc_b = tokenizer(
                ["paraphrase: " + t for t in batch],
                return_tensors="pt", padding=True,
                truncation=True, max_length=96
            ).to(device)
            with torch.no_grad():
                gen_ids = model.generate(
                    input_ids=enc_b["input_ids"],
                    attention_mask=enc_b["attention_mask"],
                    max_new_tokens=48, do_sample=False,
                    decoder_start_token_id=0
                )
            paras = tokenizer.batch_decode(gen_ids, skip_special_tokens=True)
            sem   = compute_l_sem(batch, paras)

            total = loss + alpha * sem
            if total.requires_grad:
                optimizer.zero_grad()
                torch.cuda.empty_cache()
                total.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            epoch_loss += total.item()

        n_batches  = max(1, len(train_data) // batch_size)
        epoch_loss = epoch_loss / n_batches
        print(f"  Epoch {epoch+1} | Loss: {epoch_loss:.4f}")
        ckpt = f"{save_path}/epoch_{epoch+1}"
        model.save_pretrained(ckpt)
        tokenizer.save_pretrained(ckpt)
        print(f"  Saved → {ckpt}")

    print(f"\n  Generating paraphrases on 200 test samples...")
    paras = generate_paraphrases(model, tokenizer, test_ai[:200])
    asr   = eval_asr(paras)
    bs    = eval_bertscore(test_ai[:200], paras)
    rl    = eval_rouge(test_ai[:200], paras)

    pd.DataFrame({
        "id":            [f"hc3_ai_{i:05d}_evaded" for i in range(len(paras))],
        "text":          paras,
        "source":        "ai",
        "attack_type":   "gradient",
        "attack_owner":  "udaiveer",
        "generator_model": "gpt3.5-turbo",
        "original_text": test_ai[:200],
    }).to_csv(f"{save_path}/evaded.csv", index=False)

    results = {"lambda": 0.0, "asr": round(asr, 2),
               "bertscore_f1": round(bs, 4), "rouge_l": round(rl, 4),
               "loss_final": round(epoch_loss, 4)}
    with open(done_file, "w") as f:
        json.dump(results, f, indent=2)
    print(f"  λ=0.0 DONE → ASR:{asr:.1f}%  RL:{rl:.4f}")
    return results

In [12]:
# ── CELL 9: Run full sweep ───────────────────────────────────
# Trains λ = 1.0 → 0.75 → 0.5 → 0.25 → 0.0
# If session dies, re-run from Cell 1 — completed λ values are automatically skipped.

import gc

# LAMBDA_VALUES = [1.0, 0.75, 0.5, 0.25, 0.0]
# all_results   = []

# for lam in LAMBDA_VALUES:
#     res = train_one_lambda(
#         lam         = lam,
#         alpha       = 0.3,
#         epochs      = 3,
#         batch_size  = 4,      # ← back to 4
#         rl_per_batch= 1,      # ← 1 text only
#         max_train   = 5000,
#     )
#     all_results.append(res)
#     gc.collect()
#     torch.cuda.empty_cache()

LAMBDA_VALUES = [1.0, 0.75, 0.5, 0.25]  # already done
all_results = []

# Load existing results from DONE.json
for lam in [1.0, 0.75, 0.5, 0.25]:
    lam_tag = str(lam).replace(".", "_")
    done = f"{SAVE_DIR}/lambda_{lam_tag}/DONE.json"
    if os.path.exists(done):
        with open(done) as f:
            all_results.append(json.load(f))
        print(f"λ={lam} loaded from cache")

# Now run λ=0.0
res = train_one_lambda_pure_rl(alpha=0.3, epochs=3, batch_size=4, max_train=5000)
all_results.append(res)

λ=1.0 loaded from cache
λ=0.75 loaded from cache
λ=0.5 loaded from cache
λ=0.25 loaded from cache

  λ = 0.0   (gradient: 0.0   RL: 1.0)


Loading weights: 100%|██████████| 257/257 [00:00<00:00, 3927.19it/s]


trainable params: 1,769,472 || all params: 224,673,024 || trainable%: 0.7876


  Epoch 1/3  λ=0.0: 100%|██████████| 1250/1250 [20:32<00:00,  1.01it/s]


  Epoch 1 | Loss: 0.2621
  Saved → NLP_λ_Sweep_Checkpoints/lambda_0_0/epoch_1


...

  Epoch 2 | Loss: 0.2811
  Saved → NLP_λ_Sweep_Checkpoints/lambda_0_0/epoch_2


  Epoch 3/3  λ=0.0: 100%|██████████| 1250/1250 [2:25:04<00:00,  6.96s/it] 


  Epoch 3 | Loss: 0.2841
  Saved → NLP_λ_Sweep_Checkpoints/lambda_0_0/epoch_3

  Generating paraphrases on 200 test samples...


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 4681.40it/s]
DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  λ=0.0 DONE → ASR:100.0%  RL:0.0055


In [13]:
# ── CELL 10: Summary + save ──────────────────────────────────
summary = pd.DataFrame(all_results).sort_values("lambda")
summary.to_csv(f"{SAVE_DIR}/lambda_sweep_results.csv", index=False)

print("\n" + "="*60)
print("FINAL λ-SWEEP SUMMARY")
print("="*60)
print(summary.to_string(index=False))

# Baseline (no attack) for reference
baseline_det = eval_asr(test_ai[:500])
print(f"\nBaseline (no evasion): {100 - baseline_det:.1f}% detected as AI")
print(f"Best ASR achieved:     {summary['asr'].max():.1f}%  (λ={summary.loc[summary['asr'].idxmax(), 'lambda']})")
print(f"Best BERTScore:        {summary['bertscore_f1'].max():.4f}")
print(f"\nResults saved → {SAVE_DIR}/lambda_sweep_results.csv")


FINAL λ-SWEEP SUMMARY
 lambda   asr  bertscore_f1  rouge_l  loss_final
   0.00 100.0        0.5597   0.0055      0.2841
   0.25 100.0       -1.0000   0.0212      0.2228
   0.50  99.5       -1.0000   0.1076      0.1824
   0.75 100.0       -1.0000   0.0001      0.2841
   1.00  97.5       -1.0000   0.1991      0.1426

Baseline (no evasion): 99.0% detected as AI
Best ASR achieved:     100.0%  (λ=0.0)
Best BERTScore:        0.5597

Results saved → NLP_λ_Sweep_Checkpoints/lambda_sweep_results.csv


In [19]:
# ── CELL 11: Iterative loop ASR@K ───────────────────────────
# Run the best λ model through K refinement rounds

best_lam = summary.loc[summary["asr"].idxmax(), "lambda"]
best_tag  = str(best_lam).replace(".", "_")
print(f"\nRunning iterative loop with best λ={best_lam}...")

# Reload best model
best_model, best_tok = build_model()
best_model.load_adapter(
    f"{SAVE_DIR}/lambda_{best_tag}/epoch_3",
    adapter_name="default"
)
best_model.set_adapter("default")
best_model.eval()

def iterative_asr(model, tokenizer, texts, K=5, threshold=0.5, batch_size=32):
    """
    For each text, refine up to K rounds until detector confidence < threshold.
    Returns ASR@k for each k.
    """
    current = list(texts)
    asr_at_k = {}

    for k in range(1, K+1):
        current = generate_paraphrases(model, tokenizer, current)

        # Check detector confidence
        still_ai = []
        successes = 0
        for i, text in enumerate(current):
            enc = det_tok(text, return_tensors="pt",
                          truncation=True, max_length=512).to(device)
            with torch.no_grad():
                ai_prob = torch.softmax(detector(**enc).logits, dim=-1)[0, 1].item()
            if ai_prob < threshold:
                successes += 1
                still_ai.append(texts[i])  # keep original for further rounds
            else:
                still_ai.append(text)      # keep refining

        asr_at_k[f"ASR@{k}"] = round(successes / len(texts) * 100, 2)
        current = still_ai
        print(f"  Round {k}: ASR@{k} = {asr_at_k[f'ASR@{k}']}%")

    return asr_at_k

asr_k = iterative_asr(best_model, best_tok, test_ai[:200], K=5)
print("\nIterative Inference Loop Results:")
for k, v in asr_k.items():
    print(f"  {k}: {v}%")

pd.DataFrame([asr_k]).to_csv(f"{SAVE_DIR}/asr_at_k.csv", index=False)
print(f"Saved → {SAVE_DIR}/asr_at_k.csv")


Running iterative loop with best λ=0.0...


Loading weights: 100%|██████████| 257/257 [00:00<00:00, 3816.70it/s]


trainable params: 1,769,472 || all params: 224,673,024 || trainable%: 0.7876
  Round 1: ASR@1 = 100.0%
  Round 2: ASR@2 = 100.0%
  Round 3: ASR@3 = 100.0%
  Round 4: ASR@4 = 100.0%
  Round 5: ASR@5 = 100.0%

Iterative Inference Loop Results:
  ASR@1: 100.0%
  ASR@2: 100.0%
  ASR@3: 100.0%
  ASR@4: 100.0%
  ASR@5: 100.0%
Saved → NLP_λ_Sweep_Checkpoints/asr_at_k.csv


In [20]:
# Add this between Cell 11 and Cell 12
import gc
del best_model
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM freed: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB available")

VRAM freed: 49.9 GB available


In [21]:
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    T5ForConditionalGeneration,
    T5Tokenizer
)

In [23]:
# ── CELL 12: CPTR — Test evaded samples against DetectGPT ───
# Reuses score_model and t5_model from earlier
# If session is fresh, reload them first:

print("Loading GPT-2 for DetectGPT scoring...")
score_tokenizer = AutoTokenizer.from_pretrained("gpt2")
score_model     = AutoModelForCausalLM.from_pretrained("gpt2").to(device)
score_model.eval()

t5_tokenizer = T5Tokenizer.from_pretrained("t5-small")
t5_model     = T5ForConditionalGeneration.from_pretrained("t5-small").to(device)
t5_model.eval()

def get_log_prob(text, max_len=128):
    inputs = score_tokenizer(text, return_tensors="pt",
                             truncation=True, max_length=max_len).to(device)
    with torch.no_grad():
        loss = score_model(**inputs, labels=inputs["input_ids"]).loss
    return -loss.item()

def perturb_text(text, n=10):
    words = text.split()
    if len(words) < 5:
        return [text] * n
    results = []
    for _ in range(n):
        masked = words.copy()
        idx = random.sample(range(len(masked)), max(1, int(len(masked)*0.15)))
        for i in idx:
            masked[i] = "<extra_id_0>"
        masked_text = " ".join(masked)
        try:
            inp = t5_tokenizer(masked_text, return_tensors="pt",
                               truncation=True, max_length=128).to(device)
            with torch.no_grad():
                out = t5_model.generate(**inp, max_new_tokens=15)
            filled = t5_tokenizer.decode(out[0], skip_special_tokens=True)
            fill_word = filled.strip().split()[0] if filled.strip() else words[idx[0]]
            result = [w if w != "<extra_id_0>" else fill_word for w in masked]
            results.append(" ".join(result))
        except:
            results.append(text)
    return results

def detectgpt_score(text, n=10):
    orig_lp = get_log_prob(text)
    perturbs = perturb_text(text, n)
    perturb_lps = [get_log_prob(p) for p in perturbs]
    return orig_lp - float(np.mean(perturb_lps))

# Compute CPTR for each λ
DGPT_THRESHOLD = 1.039   # same recalibrated threshold from Eval 2
DGPT_N         = 100     # 100 samples per λ (faster)
cptr_results   = []

for lam in LAMBDA_VALUES:
    lam_tag   = str(lam).replace(".", "_")
    evaded_path = f"{SAVE_DIR}/lambda_{lam_tag}/evaded.csv"

    if not os.path.exists(evaded_path):
        print(f"λ={lam}: no evaded.csv found, skipping")
        continue

    evaded_df  = pd.read_csv(evaded_path)
    evaded_texts = [
        str(t) for t in evaded_df["text"].tolist()
        if isinstance(t, str) and len(t.strip()) > 0
    ][:DGPT_N]

    print(f"\nDetectGPT on λ={lam} evaded samples ({DGPT_N} texts)...")
    scores = [detectgpt_score(t) for t in tqdm(evaded_texts)]

    # CPTR = % evaded texts that also fool DetectGPT
    fooled = sum(s < DGPT_THRESHOLD for s in scores)
    cptr   = fooled / len(scores) * 100

    cptr_results.append({
        "lambda": lam,
        "cptr_detectgpt": round(cptr, 2),
        "avg_dgpt_score": round(float(np.mean(scores)), 4),
    })
    print(f"  λ={lam} → CPTR (DetectGPT): {cptr:.1f}%")

Loading GPT-2 for DetectGPT scoring...


Loading weights: 100%|██████████| 131/131 [00:00<00:00, 5268.06it/s]



DetectGPT on λ=1.0 evaded samples (100 texts)...


100%|██████████| 100/100 [01:41<00:00,  1.01s/it]


  λ=1.0 → CPTR (DetectGPT): 53.0%

DetectGPT on λ=0.75 evaded samples (100 texts)...


...

  λ=0.75 → CPTR (DetectGPT): 18.0%

DetectGPT on λ=0.5 evaded samples (100 texts)...


100%|██████████| 100/100 [01:42<00:00,  1.02s/it]


  λ=0.5 → CPTR (DetectGPT): 87.0%

DetectGPT on λ=0.25 evaded samples (100 texts)...


100%|██████████| 100/100 [00:19<00:00,  5.07it/s]

  λ=0.25 → CPTR (DetectGPT): 63.0%


In [24]:
# ── CELL 12: Full results table ──────────────────────────────
summary  = pd.read_csv(f"{SAVE_DIR}/lambda_sweep_results.csv")
cptr_df  = pd.DataFrame(cptr_results)
final_df = summary.merge(cptr_df, on="lambda", how="left")

# Add baseline row (no attack)
baseline_row = {
    "lambda": "none",
    "asr": round(100 - 98.9, 2),   # 1.1% escape without evasion
    "cptr_detectgpt": round(100 - 87.5, 2),  # 12.5% escape without evasion
    "bertscore_f1": 1.0,
    "rouge_l": 1.0,
    "loss_final": None,
}

print("\n" + "="*70)
print("FINAL RESULTS — λ SWEEP + CPTR")
print("="*70)
print(final_df[["lambda", "asr", "cptr_detectgpt",
                "bertscore_f1", "rouge_l"]].to_string(index=False))
print("\nBaseline (no evasion): ASR=1.1%  CPTR=12.5%")

final_df.to_csv(f"{SAVE_DIR}/final_results.csv", index=False)
print(f"\nSaved → {SAVE_DIR}/final_results.csv")


FINAL RESULTS — λ SWEEP + CPTR
 lambda   asr  cptr_detectgpt  bertscore_f1  rouge_l
   0.00 100.0             NaN        0.5597   0.0055
   0.25 100.0            63.0       -1.0000   0.0212
   0.50  99.5            87.0       -1.0000   0.1076
   0.75 100.0            18.0       -1.0000   0.0001
   1.00  97.5            53.0       -1.0000   0.1991

Baseline (no evasion): ASR=1.1%  CPTR=12.5%

Saved → NLP_λ_Sweep_Checkpoints/final_results.csv


In [26]:
# ── CELL 13: Fix semantic scores for λ=1.0 and λ=0.5 ────────
# sem_tok and sem_model are already loaded from Cell 6 — no reload needed

def eval_bertscore_fixed(originals, paraphrases):
    enc_o = sem_tok(originals,   padding=True, truncation=True,
                    max_length=256, return_tensors="pt").to(device)
    enc_p = sem_tok(paraphrases, padding=True, truncation=True,
                    max_length=256, return_tensors="pt").to(device)
    with torch.no_grad():
        emb_o = mean_pool(sem_model(**enc_o), enc_o["attention_mask"])
        emb_p = mean_pool(sem_model(**enc_p), enc_p["attention_mask"])
    return F.cosine_similarity(emb_o, emb_p).mean().item()

print("Semantic similarity (MiniLM cosine) for clean λ configs:\n")
for lam in [1.0, 0.5]:
    lam_tag   = str(lam).replace(".", "_")
    evaded_df = pd.read_csv(f"{SAVE_DIR}/lambda_{lam_tag}/evaded.csv")
    paras     = [str(t) for t in evaded_df["text"].tolist()[:200] if str(t).strip()]
    originals = [str(t) for t in evaded_df["original_text"].tolist()[:200] if str(t).strip()]
    min_len   = min(len(paras), len(originals))
    paras, originals = paras[:min_len], originals[:min_len]

    bs = eval_bertscore_fixed(originals, paras)
    rl = eval_rouge(originals, paras)
    print(f"  λ={lam} → Semantic similarity: {bs:.4f}  ROUGE-L: {rl:.4f}")

Semantic similarity (MiniLM cosine) for clean λ configs:

  λ=1.0 → Semantic similarity: 0.0968  ROUGE-L: 0.0932
  λ=0.5 → Semantic similarity: 0.1612  ROUGE-L: 0.0705


In [27]:
# ── CELL 14: Print clean readable examples ───────────────────
# Show best λ=0.5 examples where evasion succeeded AND text is readable

evaded_df = pd.read_csv(f"{SAVE_DIR}/lambda_0_5/evaded.csv")

# Filter: only rows where text is non-empty and reasonably long
evaded_df["text"]          = evaded_df["text"].astype(str)
evaded_df["original_text"] = evaded_df["original_text"].astype(str)

clean = evaded_df[
    (evaded_df["text"].str.len() > 80) &
    (evaded_df["original_text"].str.len() > 80) &
    (~evaded_df["text"].str.startswith("entail")) &
    (~evaded_df["text"].str.startswith("аа")) &
    (~evaded_df["text"].str.startswith("nn")) &
    (~evaded_df["text"].str.startswith("True")) &
    (~evaded_df["text"].str.startswith("negative")) &
    (~evaded_df["text"].str.contains("ааааа")) &
    (~evaded_df["text"].str.contains("nnnnn"))
].reset_index(drop=True)

print(f"Clean readable samples: {len(clean)} / {len(evaded_df)}\n")

# Show top 5 examples
for i in range(min(8, len(clean))):
    print("="*70)
    print(f"EXAMPLE {i+1}")
    print("-"*35)
    print(f"ORIGINAL (AI-generated):\n{clean.loc[i, 'original_text'][:400]}")
    print(f"\nEVADED (humanized):\n{clean.loc[i, 'text'][:400]}")
    print()

Clean readable samples: 78 / 200

EXAMPLE 1
-----------------------------------
ORIGINAL (AI-generated):
Halloween is a holiday that is celebrated in many European countries, although the way it is celebrated can vary from one country to another. In some European countries, Halloween is a very popular holiday and is celebrated with costumes, parties, and activities such as trick-or-treating, which is when children dress up in costumes and go door-to-door asking for candy. In other European countries,

EVADED (humanized):
some people in some countries celebrate the holiday with parties and other activities .

EXAMPLE 2
-----------------------------------
ORIGINAL (AI-generated):
Unions are organizations that represent the collective interests of workers. They are made up of workers who come together to negotiate with their employer for things like better pay, safer working conditions, and better benefits. Unions have the authority to negotiate on behalf of their members because the work